In [ ]:
import wrds
import pandas as pd
from pathlib import Path
import pyarrow

In [ ]:
PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_RAW.mkdir(parents=True, exist_ok=True)
START = '2004-01-01'
END = '2027-06-30' 

In [ ]:
db = wrds.Connection()

In [ ]:
def pull_and_save(query, filename, overwrite=False):
    """Pull from WRDS and save as parquet. Skips if file already exists."""
    filepath = DATA_RAW / filename
    if filepath.exists() and not overwrite:
        size_mb = filepath.stat().st_size / 1e6
        print(f"  ✓ Already exists: {filename} ({size_mb:.1f} MB) — skipping")
        return pd.read_parquet(filepath)
    
    print(f"  Pulling: {filename} ...")
    df = db.raw_sql(query)
    df.to_parquet(filepath, engine='pyarrow', compression='snappy')
    size_mb = filepath.stat().st_size / 1e6
    print(f"  ✓ Saved: {len(df):,} rows, {size_mb:.1f} MB")
    return df

In [ ]:
#ibes, crsp, comp, ciqsamp_transcripts, ff/ff_all, crsp_q_ccm
#comp_snapshot for look ahead bias
print("My libraries:", db.list_libraries())
print("\nIBES tables:", db.list_tables(library='ibes'))
print("\nCRSP tables:", db.list_tables(library='crsp'))
print("\nCRSP_Q_CCM tables:", db.list_tables(library='crsp_q_ccm'))
print("\nCompustat tables:", db.list_tables(library='comp'))
print("\nCapital IQ tables:", db.list_tables(library='ciqsamp_transcripts'))
print("\nFF tables:", db.list_tables(library='ff'))
print("\nFF_all tables:", db.list_tables(library='ff_all'))

In [ ]:
tables_to_check = [
    ('ibes', 'statsumu_epsus'),
    ('ibes', 'actu_epsus'),
    ('ibes', 'idsum'),
    ('crsp', 'dsf'),
    ('crsp', 'dsenames'),
    ('crsp_q_ccm', 'ccmxpf_lnkhist'),
    ('comp', 'fundq'),
    ('ff', 'fivefactors_daily'),
    ('ff', 'factors_daily'),
]

for lib, tbl in tables_to_check:
    print(f"\n {lib}.{tbl}")
    desc = db.describe_table(library=lib, table=tbl)
    print(desc[['name', 'type']].to_string())

In [ ]:
#Format:
#SQL_Query_name = f"""
# SQL QUERY ACTUAL
#"""
#name_summary (df) = pull_and_save(SQL_QUERY_NAME,'SQL_QUERY_NAME.parquet')

***IBES***

In [ ]:
#IBES Unadjusted Summary
ibes_summary_q = f"""
SELECT ticker, cusip, oftic, cname,
       statpers, fpedats,
       measure, fiscalp, fpi, estflag, curcode,
       numest, numup, numdown,
       medest, meanest, stdev, highest, lowest,
       usfirm
FROM ibes.statsumu_epsus
WHERE measure = 'EPS'
  AND fpi = '6'
  AND statpers BETWEEN '{START}' AND '{END}'
"""
ibes_summary = pull_and_save(ibes_summary_q, 'ibes_summary.parquet', overwrite=True)

In [ ]:
#IBES Unadjusted Actuals
ibes_actuals_q = f"""
SELECT ticker, cusip, oftic, cname,
       pends, measure, pdicity,
       anndats, anntims, actdats, acttims,
       value, curr_act, usfirm
FROM ibes.actu_epsus
WHERE measure = 'EPS'
  AND pends BETWEEN '{START}' AND '{END}'
"""
ibes_actuals = pull_and_save(ibes_actuals_q, 'ibes_actuals.parquet',overwrite=True)

In [ ]:
#IBES Identifiers
ibes_id_q = """
SELECT ticker, cusip, oftic, cname,
       dilfac, pdi, ccopcf, tnthfac, instrmnt,
       exchcd, country, compflag, usfirm, sdates
FROM ibes.idsum
"""
ibes_id = pull_and_save(ibes_id_q, 'ibes_identifiers.parquet',overwrite=True)

***CRSP***

In [ ]:
#CRSP Daily Stock File
crsp_dsf_q = f"""
SELECT a.permno, a.permco, a.date,
       a.bidlo, a.askhi, a.prc, a.vol, a.ret, a.bid, a.ask,
       a.shrout, a.cfacpr, a.cfacshr, a.openprc, a.numtrd, a.retx,
       a.cusip, a.hexcd, a.hsiccd, a.issuno,
       b.namedt, b.nameendt, b.shrcd, b.exchcd, b.siccd,
       b.ncusip, b.ticker, b.comnam, b.shrcls, b.tsymbol,
       b.naics, b.primexch, b.trdstat, b.secstat
FROM crsp.dsf AS a
LEFT JOIN crsp.dsenames AS b
  ON a.permno = b.permno
  AND b.namedt <= a.date
  AND a.date <= b.nameendt
WHERE a.date BETWEEN '{START}' AND '{END}'
  AND b.shrcd IN (10, 11)
  AND b.exchcd IN (1, 2, 3)
"""
crsp_dsf = pull_and_save(crsp_dsf_q, 'crsp_dsf.parquet',overwrite=True)

In [ ]:
#CRSP Compustat-linking
ccm_link_q = """
SELECT gvkey, linkprim, liid, linktype,
       lpermno, lpermco, linkdt, linkenddt
FROM crsp_q_ccm.ccmxpf_lnkhist
WHERE linktype IN ('LU', 'LC')
  AND linkprim IN ('P', 'C')
"""
ccm_link = pull_and_save(ccm_link_q, 'ccm_link.parquet')

In [ ]:
#Compustat Quaterly
comp_fundq_q = f"""
SELECT gvkey, datadate, rdq, fyearq, fqtr, fyr,
       indfmt, consol, popsrc, datafmt,
       tic, cusip, conm, cik, exchg, costat,
       epspxq, epsfxq, epspiq, epsfiq,
       saleq, revtq, niq, ibq,
       atq, ltq, ceqq, seqq, teqq,
       cshoq, cshprq, cshfdq,
       prccq, prchq, prclq, mkvaltq,
       dvpspq, dvpsxq
FROM comp.fundq
WHERE datadate BETWEEN '{START}' AND '{END}'
  AND indfmt = 'INDL'
  AND datafmt = 'STD'
  AND popsrc = 'D'
  AND consol = 'C'
"""
comp_fundq = pull_and_save(comp_fundq_q, 'compustat_fundq.parquet')

In [ ]:
#FF 5 factor & momentum
ff_q = f"""
SELECT date, mktrf, smb, hml, rmw, cma, umd, rf
FROM ff.fivefactors_daily
WHERE date BETWEEN '{START}' AND '{END}'
"""
ff_factors = pull_and_save(ff_q, 'ff_factors_daily.parquet')

**TRANSCRIPT DATA THROUGH FMP**

The WRDS subscription I have is only a sample of 20 transcripts, so I have subscribed to FMP and will pull the US earning call transcript data from there. To see why only US companies, refer to the documentation.

In [ ]:
import requests
import os
import json
import time
from pathlib import Path
from datetime import datetime

In [ ]:
PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_RAW.mkdir(parents=True, exist_ok=True)
TRANSCRIPTS_DIR = DATA_RAW / 'transcripts'
NEWS_DIR = DATA_RAW / 'news'
PRESS_DIR = DATA_RAW / 'press_releases'
INSIDER_DIR = DATA_RAW / 'insider_trading'
INST_DIR = DATA_RAW / 'institutional'
for d in [TRANSCRIPTS_DIR, NEWS_DIR, PRESS_DIR, INSIDER_DIR, INST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# API key
FMP_API_KEY = os.environ.get('FMP_API_KEY')

FMP_BASE = "https://financialmodelingprep.com/stable"

In [ ]:
def fetch_with_retry(url, params, max_retries=5, timeout=30):
    """GET with exponential backoff on rate limits and transient errors."""
    params = {**params, 'apikey': FMP_API_KEY}
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code == 200:
                return r.json()
            elif r.status_code == 429:  # rate limited
                wait = 2 ** attempt
                print(f"  Rate limited, sleeping {wait}s...")
                time.sleep(wait)
            elif r.status_code in (403, 404):
                return None  # not authorized or not found - skip
            else:
                print(f"  HTTP {r.status_code} on {url}: {r.text[:200]}")
                time.sleep(1 + attempt)
        except requests.exceptions.RequestException as e:
            print(f"  Network error: {e}")
            time.sleep(2 ** attempt)
    return None


def save_json(filepath, data):
    """Idempotent JSON save. Returns 'skipped' or 'saved'."""
    filepath = Path(filepath)
    if filepath.exists():
        return 'skipped'
    filepath.parent.mkdir(parents=True, exist_ok=True)
    filepath.write_text(json.dumps(data))
    return 'saved'

In [ ]:
def coverage_report(symbols=BENCHMARKS):
    rows = []
    for sym in symbols:
        dates = fetch_with_retry(
            f"{FMP_BASE}/earning-call-transcript-dates",
            {'symbol': sym}
        )
        if dates:
            years = [int(d['fiscalYear']) for d in dates]   # was 'year'
            rows.append({
                'symbol': sym,
                'count': len(dates),
                'earliest': min(years),
                'latest': max(years),
            })
        else:
            rows.append({'symbol': sym, 'count': 0, 'earliest': None, 'latest': None})
    return pd.DataFrame(rows)

In [ ]:
def get_transcript_dates(symbol):
    """Get list of {fiscalYear, quarter, date} dicts with transcripts for a symbol."""
    return fetch_with_retry(
        f"{FMP_BASE}/earning-call-transcript-dates",
        {'symbol': symbol}
    ) or []

def pull_one_transcript(symbol, year, quarter):
    """Pull and save one transcript. Returns 'skipped', 'saved', or 'missing'."""
    filepath = TRANSCRIPTS_DIR / symbol / f"{year}Q{quarter}.json"
    if filepath.exists():
        return 'skipped'
    data = fetch_with_retry(
        f"{FMP_BASE}/earning-call-transcript",
        {'symbol': symbol, 'year': year, 'quarter': quarter}
    )
    if data:
        return save_json(filepath, data)
    return 'missing'

def bulk_pull_transcripts(symbols, progress_every=25):
    stats = {'saved': 0, 'skipped': 0, 'missing': 0, 'errors': 0}
    start_time = time.time()
    for i, sym in enumerate(symbols):
        try:
            dates = get_transcript_dates(sym)
            for d in dates:
                result = pull_one_transcript(sym, d['fiscalYear'], d['quarter'])
                stats[result] += 1
        except Exception as e:
            print(f"  Error on {sym}: {e}")
            stats['errors'] += 1
        if (i + 1) % progress_every == 0:
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed
            eta_min = (len(symbols) - (i + 1)) / rate / 60
            print(f"[{i+1}/{len(symbols)}] {sym} | {stats} | ETA {eta_min:.0f}min")
    return stats

In [ ]:
# Universe
universe = fetch_with_retry(f"{FMP_BASE}/earnings-transcript-list", {})
universe_df = pd.DataFrame(universe)
universe_df.to_parquet(DATA_RAW / 'fmp_universe.parquet')
print(f"Universe: {len(universe_df)} firms")
print(f"Columns: {universe_df.columns.tolist()}")
print(universe_df.head())

SYMBOLS = sorted(universe_df['symbol'].unique().tolist())

In [ ]:
# Filter to US-listed symbols (no exchange suffix)
us_symbols = [s for s in SYMBOLS if '.' not in s]
print(f"US universe: {len(us_symbols)} firms (filtered from {len(SYMBOLS)})")
print(f"Sample: {us_symbols[:20]}")

In [ ]:
final_stats = bulk_pull_transcripts(us_symbols, progress_every=25)